<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>51095 מודלי יסוד: ארכיטקטורות, אימון ויישור</p>
<p>מטלת בית 2 — LLM Alignment</p>
<p>שם: נדב פיירמן שטרן</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מבנה המחברת:</p>
<ul>
<li>חלק א׳ — SFT מלא על 3,000 דוגמאות Dolly-15k, 12 פרומפטים קבועים לפני ואחרי</li>
<li>חלק ב׳ — Full FT מול LoRA מול QLoRA: פרמטרים, זיכרון, זמן ואיכות</li>
<li>חלק ג׳ — 200 דוגמאות בלבד, ו-3,000 שב-30% מהן התשובה הוחלפה</li>
<li>חלק ד׳ — 300 פרומפטים, שתי דגימות לכל אחד, שופט זוגי ו-reward model</li>
<li>חלק ה׳ — DPO מול SFT, עם שופט שני לבדיקת מעגליות</li>
<li>חלק ו׳ — Best-of-N עם N=4 ו-N=8 מול DPO, איכות מול עלות</li>
<li>סינתזה, ובונוס: LoRA על attention בלבד מול כל השכבות</li>
</ul>
<p>מודל: Qwen2.5-0.5B base. שופט: Phi-4, 14B ב-NF4. שופט שני: Granite-3.3-8B. הדו"ח בתוך המחברת, לפי הנחיית המרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תקציר מנהלים:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>עקרונות הניסוי:</p>
<ul>
<li>משתנה אחד בכל השוואה: אותו בסיס, דאטה, epochs, אורך רצף, optimizer ו-seed</li>
<li>קצב הלמידה שונה בין Full FT ל-LoRA, לפי הערך המקובל לכל שיטה</li>
<li>כל השוואת שופט רצה בשני seeds של דגימה ובשני סדרי הצגה</li>
<li>המדדים והבדיקות הוגדרו לפני ההרצה</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>התקנת ספריות בגרסאות נעוצות.</p>
<ul>
<li>torchao מוסר אחרי ההתקנה: peft 0.20 נופל מול הגרסה של Colab, והמחברת לא משתמשת בו</li>
<li>אם Colab מבקש restart, לאשר ולהריץ מההתחלה</li>
</ul>
</div>

In [ ]:
!pip install -q -U "transformers==5.16.1" "trl==1.13.0" "peft==0.20.0" "datasets==5.0.1" "accelerate==1.15.0" "bitsandbytes>=0.47" sentencepiece scikit-learn
!pip uninstall -y -q torchao

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>כל הייבואים של המחברת, בתא אחד:</p>
<ul>
<li>torch, transformers, peft, trl, datasets — מודלים ואימון</li>
<li>numpy, pandas, sklearn, matplotlib — טבלאות, reward model וגרפים</li>
<li>bitsandbytes — ל-QLoRA. google.colab ו-openai — לשופט API אופציונלי. שלושתם ב-try, כי ייתכן שאינם מותקנים</li>
<li>peft.import_utils — מיד אחרי הייבוא, בדיקת torchao של peft מוחלפת ב-False, למקרה שהסרת torchao לא נקלטה</li>
</ul>
</div>

In [ ]:
import os, sys, gc, re, json, time, math, random, warnings, textwrap
import importlib.metadata as im
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import transformers, datasets, peft, trl
import peft.import_utils as _peft_import_utils
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset, Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer, DPOConfig, DPOTrainer
from huggingface_hub import model_info, dataset_info
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

try:
    import bitsandbytes as bnb
    HAS_BNB = True
except Exception:
    HAS_BNB = False
try:
    import google.colab
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
try:
    from openai import OpenAI
except Exception:
    OpenAI = None

_peft_import_utils.is_torchao_available.cache_clear()
for _name, _mod in list(sys.modules.items()):
    if _name.split(".")[0] == "peft" and hasattr(_mod, "is_torchao_available"):
        _mod.is_torchao_available = lambda: False

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הגדרות סביבה: זיכרון CUDA, השתקת אזהרות, ועיצוב טבלאות וגרפים.</p>
</div>

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
plt.style.use("ggplot")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הגדרות ההרצה, במקום אחד:</p>
<ul>
<li>BASE_MODEL — Qwen2.5-0.5B base</li>
<li>JUDGE_MODEL — Phi-4, ממשפחה אחרת כנדרש. JUDGE2_MODEL — Granite-3.3-8B</li>
<li>CFG — גדלי דאטה, epochs, batch, קצבי למידה, LoRA, DPO ו-Best-of-N</li>
<li>SMOKE — הרצה מוקטנת על CPU לבדיקת הקוד בלבד</li>
</ul>
</div>

In [ ]:
SMOKE = os.environ.get("HW2_SMOKE", "0") == "1"

GLOBAL_SEED  = 51095
BASE_MODEL   = "Qwen/Qwen2.5-0.5B"
DATASET      = "databricks/databricks-dolly-15k"
JUDGE_MODEL  = "HuggingFaceTB/SmolLM2-360M-Instruct" if SMOKE else "microsoft/phi-4"
JUDGE2_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct" if SMOKE else "ibm-granite/granite-3.3-8b-instruct"
JUDGE_4BIT   = {"microsoft/phi-4"}

JUDGE_API = {k: os.environ.get(k) for k in ("JUDGE_BASE_URL", "JUDGE_API_KEY", "JUDGE_API_MODEL")}
if IN_COLAB:
    for k in JUDGE_API:
        try:
            v = userdata.get(k)
            if v:
                JUDGE_API[k] = v
        except Exception:
            pass
USE_API_JUDGE = all(JUDGE_API.values())

CFG = dict(
    n_train=3000, n_eval=200, n_small=200, noise_rate=0.30,
    epochs=2, max_len=512, bsz=8, grad_acc=2, lr_full=2e-5, lr_lora=2e-4,
    lora_r=16, lora_alpha=32, lora_dropout=0.05,
    max_new=256, n_pref_prompts=300, n_pref_extra=70, min_pairs=40, pref_temp=0.8, pref_top_p=0.95, n_pref_holdout=30,
    dpo_beta=0.1, dpo_epochs=3, dpo_lr=5e-5, dpo_bsz=4, dpo_grad_acc=2,
    bon_N=8, eval_temp=0.7, eval_seeds=[1, 2], n_judge=50,
    judge_max_new=220, judge_bsz=8, judge_max_chars=1500,
)
if SMOKE:
    CFG.update(n_train=8, n_eval=4, n_small=4, epochs=1, max_len=160, bsz=2, grad_acc=1,
               max_new=24, n_pref_prompts=6, n_pref_extra=2, min_pairs=99, n_pref_holdout=2, dpo_epochs=1, dpo_bsz=2, dpo_grad_acc=1,
               bon_N=4, eval_seeds=[1], n_judge=3, judge_max_new=140, judge_bsz=4)

RUN_DIR = os.path.abspath("hw2_runs")
os.makedirs(RUN_DIR, exist_ok=True)
print("SMOKE" if SMOKE else "FULL RUN", "| judge:", JUDGE_API["JUDGE_API_MODEL"] if USE_API_JUDGE else JUDGE_MODEL, "| run dir:", RUN_DIR)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>seed, זיהוי חומרה ותיעוד הסביבה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>set_all_seeds: קובעת seed לכל מקורות האקראיות.</p>
</div>

In [ ]:
def set_all_seeds(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>קביעת seed, dtype לפי הכרטיס, והקטנת batch אוטומטית בכרטיס קטן מ-20 GiB.</p>
</div>

In [ ]:
set_all_seeds()

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
BF16    = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
DTYPE   = torch.bfloat16 if BF16 else (torch.float16 if DEVICE == "cuda" else torch.float32)
GPU_GIB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if DEVICE == "cuda" else 0.0
if DEVICE == "cuda" and GPU_GIB < 20 and not SMOKE:
    CFG.update(bsz=4, grad_acc=4)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>hub_sha: מזהה ה-commit של מודל או דאטהסט ב-Hub, לשחזוריות.</p>
</div>

In [ ]:
def hub_sha(repo, kind="model"):
    try:
        return (dataset_info(repo) if kind == "dataset" else model_info(repo)).sha
    except Exception as e:
        return f"unavailable ({type(e).__name__})"

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תיעוד הסביבה והגרסאות ב-RESULTS.</p>
</div>

In [ ]:
ENV = {
    "torch": torch.__version__, "transformers": transformers.__version__, "trl": trl.__version__,
    "peft": peft.__version__, "datasets": datasets.__version__,
    "bitsandbytes": (bnb.__version__ if HAS_BNB else "not installed"),
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
    "gpu_total_gib": GPU_GIB, "dtype": str(DTYPE).replace("torch.", ""), "seed": GLOBAL_SEED, "smoke": SMOKE,
}
RESULTS = {
    "environment": ENV, "config": dict(CFG),
    "models": {"base": {"name": BASE_MODEL, "revision": hub_sha(BASE_MODEL)},
               "judge": {"name": JUDGE_API["JUDGE_API_MODEL"] if USE_API_JUDGE else JUDGE_MODEL,
                         "revision": "api" if USE_API_JUDGE else hub_sha(JUDGE_MODEL)},
               "judge2": {"name": JUDGE2_MODEL, "revision": hub_sha(JUDGE2_MODEL)},
               "dataset": {"name": DATASET, "revision": hub_sha(DATASET, "dataset")}},
    "runs": {}, "comparisons": {}, "judge_log": {},
}
print(" | ".join(f"{k}={v}" for k, v in ENV.items()))
print(json.dumps(RESULTS["models"], indent=1))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>טוקנייזר ותבנית שיחה:</p>
<ul>
<li>תבנית ChatML של Qwen נכפית על מודל ה-base</li>
<li>eos הוא im_end, סוף התור של העוזר. pad הוא endoftext</li>
</ul>
</div>

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.eos_token = "<|im_end|>"
tok.pad_token = "<|endoftext|>"
tok.padding_side = "right"
IM_END, ENDTEXT = tok.convert_tokens_to_ids("<|im_end|>"), tok.convert_tokens_to_ids("<|endoftext|>")
SYSTEM_PROMPT = "You are a helpful assistant."

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>to_prompt: הוראה לפרומפט ChatML עם הודעת system קבועה.</p>
</div>

In [ ]:
def to_prompt(instruction, context=""):
    user = instruction.strip()
    if context and context.strip():
        user += "\n\n" + context.strip()
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>free_gpu: משחררת זיכרון ומאפסת את מונה השיא לפני כל מדידה.</p>
</div>

In [ ]:
def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>gib: בתים ל-GiB.</p>
</div>

In [ ]:
def gib(x):
    return round(x / 2**30, 3)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>load_base: טוענת את הבסיס ב-dtype נתון, או ב-NF4 ל-QLoRA.</p>
</div>

In [ ]:
def load_base(dtype=None, quant=None):
    kw = {"dtype": dtype or DTYPE}
    if DEVICE == "cuda":
        kw["device_map"] = {"": 0}
    if quant == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
    m.config.pad_token_id = tok.pad_token_id
    return m

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בדיקת התבנית על הוראה אחת.</p>
</div>

In [ ]:
print(repr(to_prompt("Say hello.")))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>embed_responses: וקטור של 896 ממדים לכל תשובה.</p>
</div>

In [ ]:
@torch.no_grad()
def generate(model, prompts, max_new=None, temperature=None, top_p=0.95, seed=GLOBAL_SEED, num_return=1, batch_size=8):
    model.eval()
    tok.padding_side = "left"
    outs, t0 = [], time.perf_counter()
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        enc = tok(batch, return_tensors="pt", padding=True).to(model.device)
        set_all_seeds(seed)
        kw = dict(max_new_tokens=max_new or CFG["max_new"], pad_token_id=tok.pad_token_id,
                  eos_token_id=[IM_END, ENDTEXT], num_return_sequences=num_return)
        if temperature:
            kw.update(do_sample=True, temperature=float(temperature), top_p=top_p, top_k=0)
        else:
            kw.update(do_sample=False)
        out = model.generate(**enc, **kw)[:, enc["input_ids"].shape[1]:]
        for row in out.tolist():
            ended = False
            for j, t in enumerate(row):
                if t in (IM_END, ENDTEXT):
                    row, ended = row[:j], True
                    break
            outs.append(dict(text=tok.decode(row, skip_special_tokens=True).strip(), n_tokens=len(row), ended=ended))
    tok.padding_side = "right"
    return outs, round(time.perf_counter() - t0, 2)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מדדים אוטומטיים, בלי שופט.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>count_params: פרמטרים לאימון וסך הפרמטרים.</p>
</div>

In [ ]:
def count_params(model):
    if hasattr(model, "get_nb_trainable_parameters"):
        tr, tot = model.get_nb_trainable_parameters()
        return int(tr), int(tot)
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return int(tr), int(sum(p.numel() for p in model.parameters()))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>wilson: שיעור ורווח סמך של 95%.</p>
</div>

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = p + z * z / (2 * n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (round(p, 3), round((c - h) / d, 3), round((c + h) / d, 3))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>words: פירוק טקסט למילים.</p>
</div>

In [ ]:
def words(t):
    return re.findall(r"\b\w+\b", t)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>rep4: שיעור רביעיות המילים שחוזרות בתשובה.</p>
</div>

In [ ]:
def rep4(t):
    w = words(t.lower())
    if len(w) < 8:
        return 0.0
    grams = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
    return round(1 - len(set(grams)) / len(grams), 3)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>lcs_ratio: כמה מתשובת האימון שוחזר, ברמת מילים.</p>
</div>

In [ ]:
def lcs_ratio(generated, reference):
    A, B = words(generated.lower()), words(reference.lower())
    if not A or not B:
        return 0.0
    prev = [0] * (len(B) + 1)
    for x in A:
        cur = [0] * (len(B) + 1)
        for j, y in enumerate(B, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return round(prev[-1] / len(B), 3)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>behavior_stats: אורך, שיעור סיום, חזרתיות ושיעור מעבר בדיקות.</p>
</div>

In [ ]:
def behavior_stats(outs, fixed=None):
    fixed = fixed or FIXED
    return dict(n=len(outs),
                mean_words=round(float(np.mean([len(words(o["text"])) for o in outs])), 1),
                median_words=float(np.median([len(words(o["text"])) for o in outs])),
                ended_rate=round(float(np.mean([o["ended"] for o in outs])), 3),
                rep4=round(float(np.mean([rep4(o["text"]) for o in outs])), 3),
                check_pass=round(float(np.mean([bool(f["check"](o["text"])) for f, o in zip(fixed, outs)])), 3))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>12 הפרומפטים הקבועים, לכל אחד בדיקה אוטומטית:</p>
<ul>
<li>נכתבו כאן ולא נלקחו מ-Dolly, כדי שלא יופיעו באימון</li>
<li>מכסים עובדה, הסבר, פורמט, סיווג, סיכום, שכתוב, חשבון, רעיונות, עצה, קוד ותרגום</li>
</ul>
</div>

In [ ]:
PASSAGE = ("The first bicycles appeared in Europe in the early nineteenth century as wooden frames with two wheels "
           "and no pedals; riders pushed themselves along with their feet. Pedals were added a few decades later, and "
           "by the 1880s the safety bicycle, with two equal-sized wheels and a chain drive, had become popular. Cheaper "
           "steel frames and pneumatic tyres then turned cycling from a hobby of the wealthy into everyday transport "
           "for workers and families.")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>_n_items: סופרת פריטי רשימה.</p>
</div>

In [ ]:
def _n_items(t):
    return len([l for l in t.splitlines() if re.match(r"^\s*(\d+[\.\)]|[-*•])\s+\S", l)])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>_sentences: מפצלת למשפטים.</p>
</div>

In [ ]:
def _sentences(t):
    return [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>_json_ok: בודקת JSON עם name=Dana ו-age=30.</p>
</div>

In [ ]:
def _json_ok(t):
    m = re.search(r"\{.*?\}", t, re.S)
    try:
        d = json.loads(m.group(0))
        return d.get("name") == "Dana" and d.get("age") == 30
    except Exception:
        return False

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>_last_line_has_80: בודקת שהמספר 80 בשורה האחרונה.</p>
</div>

In [ ]:
def _last_line_has_80(t):
    lines = [l for l in t.strip().splitlines() if l.strip()]
    return bool(lines) and bool(re.search(r"\b80\b", lines[-1]))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הגדרת הפרומפטים והבדיקות.</p>
</div>

In [ ]:
FIXED = [
    dict(id="capital", cat="factual", prompt="What is the capital of Australia? Answer in one sentence.",
         check=lambda r: "canberra" in r.lower() and len(_sentences(r)) <= 2),
    dict(id="sky", cat="explanation", prompt="Explain in simple terms why the sky is blue, in no more than four sentences.",
         check=lambda r: 1 <= len(_sentences(r)) <= 4 and "scatter" in r.lower()),
    dict(id="fruits", cat="format", prompt="List exactly five fruits that are red, as a numbered list with one fruit per line and nothing else.",
         check=lambda r: _n_items(r) == 5),
    dict(id="json", cat="format", prompt='Return a JSON object with the keys "name" and "age" for a fictional person named Dana who is 30 years old. Output only the JSON.',
         check=_json_ok),
    dict(id="sentiment", cat="classification", prompt='Classify the sentiment of this review as Positive, Negative or Neutral: "The battery died after two hours and support never replied." Answer with one word only.',
         check=lambda r: r.strip().strip(".").lower() == "negative"),
    dict(id="summary", cat="summarization", prompt=PASSAGE + "\n\nSummarize the passage above in one sentence.",
         check=lambda r: len(words(r)) <= 40 and len(_sentences(r)) == 1),
    dict(id="formal", cat="rewrite", prompt='Rewrite the following sentence in a formal tone: "hey, can u send me the file asap?"',
         check=lambda r: len(words(r)) <= 40 and " u " not in " " + r.lower() + " " and "asap" not in r.lower()),
    dict(id="speed", cat="reasoning", prompt="A train travels 60 km in 45 minutes. What is its average speed in km/h? Show your steps briefly and put only the final number on the last line.",
         check=_last_line_has_80),
    dict(id="bakery", cat="brainstorm", prompt="Give three creative names for a bakery that sells only sourdough bread, as a numbered list, with a one-line explanation for each name.",
         check=lambda r: _n_items(r) == 3),
    dict(id="tips", cat="advice", prompt="I have trouble waking up early. Give me exactly three practical tips as a numbered list.",
         check=lambda r: _n_items(r) == 3),
    dict(id="fib", cat="code", prompt="Write a Python function fib(n) that returns the n-th Fibonacci number iteratively. Output only the code, no explanation.",
         check=lambda r: "def fib" in r and "return" in r and not re.match(r"(?i)\s*(here|this|the function|sure)", r)),
    dict(id="french", cat="translation", prompt='Translate to French: "The library closes at eight tonight."',
         check=lambda r: "biblioth" in r.lower() and "ferme" in r.lower()),
]
if SMOKE:
    FIXED = FIXED[:3]
FIXED_PROMPTS = [to_prompt(f["prompt"]) for f in FIXED]
INSTR = [f["prompt"] for f in FIXED]
print(len(FIXED), "fixed prompts:", [f["id"] for f in FIXED])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>השופט, לפי נספח ב׳:</p>
<ul>
<li>ממשפחה אחרת מ-Qwen</li>
<li>נימוק לפני פסק, בפורמט JSON, greedy</li>
<li>כל זוג בשני סדרים. פסק שמתהפך עם הסדר נספר כתיקו</li>
<li>פסק לא קריא נשאל שוב פעם אחת</li>
<li>בהרצה מקדימה Phi-3.5-mini בחר במשבצת הראשונה ב-89% מהפסקים, ולכן הוחלף ב-Phi-4</li>
</ul>
</div>

In [ ]:
JUDGE_SYSTEM = ("You are a strict, impartial evaluator of assistant responses. You compare two responses to the same "
                "instruction. Judge how well each response follows the instruction, its factual correctness, helpfulness "
                "and clarity. Do not prefer a response because it is longer, and do not prefer a response because of its position.")

JUDGE_TEMPLATE = """[Instruction]
{instruction}

[Response A]
{a}

[Response B]
{b}

Write a short rationale (2-4 sentences) comparing the two responses, then give your verdict.
Answer with a single JSON object and nothing else, in exactly this form:
{{"rationale": "<your reasoning>", "verdict": "A" or "B" or "tie"}}
Use "tie" only when the responses are of genuinely equal quality.

Here is the shape of a correct answer, for format only:
{{"rationale": "A follows the instruction and stops there, while B adds unrelated text after the answer.", "verdict": "A"}}"""

RETRY_SUFFIX = ('\n\nYour previous answer was not a single JSON object. Reply again with nothing but the JSON object, '
                'starting with {"rationale" and ending with }.')

_JUDGE = {"model": None, "tok": None, "name": None}
JUDGE_CACHE = {}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>judge_name: שם השופט הפעיל.</p>
</div>

In [ ]:
def judge_name(name=None):
    return name or (("api:" + JUDGE_API["JUDGE_API_MODEL"]) if USE_API_JUDGE else JUDGE_MODEL)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>free_judge: מוציאה את השופט מהזיכרון לפני אימון.</p>
</div>

In [ ]:
def free_judge():
    if _JUDGE["model"] is not None:
        _JUDGE.update(model=None, tok=None, name=None)
        free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>get_judge: טוענת שופט, ועוצרת אם הוא ממשפחת Qwen. Phi-4 נטען ב-NF4.</p>
</div>

In [ ]:
def get_judge(name):
    assert "qwen" not in name.lower(), "Appendix B: the judge must come from a different model family than Qwen"
    if _JUDGE["name"] != name:
        free_judge()
        jt = AutoTokenizer.from_pretrained(name)
        jt.padding_side = "left"
        if jt.pad_token is None:
            jt.pad_token = jt.eos_token
        kw = {"dtype": DTYPE}
        if DEVICE == "cuda":
            kw["device_map"] = {"": 0}
            if name in JUDGE_4BIT:
                kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
        jm = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
        _JUDGE.update(model=jm, tok=jt, name=name)
    return _JUDGE["model"], _JUDGE["tok"]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>embed_responses: וקטור של 896 ממדים לכל תשובה.</p>
</div>

In [ ]:
@torch.no_grad()
def judge_local(texts, name):
    jm, jt = get_judge(name)
    outs = []
    for i in range(0, len(texts), CFG["judge_bsz"]):
        chats = [jt.apply_chat_template([{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": t}],
                                        tokenize=False, add_generation_prompt=True, enable_thinking=False)
                 for t in texts[i:i + CFG["judge_bsz"]]]
        add_special = not (jt.bos_token and chats[0].startswith(jt.bos_token))
        enc = jt(chats, return_tensors="pt", padding=True, add_special_tokens=add_special).to(jm.device)
        out = jm.generate(**enc, max_new_tokens=CFG["judge_max_new"], do_sample=False, pad_token_id=jt.pad_token_id)
        outs += [jt.decode(r[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for r in out]
    return outs

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>judge_api: שופט API אופציונלי. לא פעיל בהרצה הזו.</p>
</div>

In [ ]:
def judge_api(texts):
    assert OpenAI is not None, "the API judge needs the openai package: pip install openai"
    assert "qwen" not in JUDGE_API["JUDGE_API_MODEL"].lower(), "Appendix B: the judge must come from a different model family"
    client = OpenAI(base_url=JUDGE_API["JUDGE_BASE_URL"], api_key=JUDGE_API["JUDGE_API_KEY"])
    outs = []
    for t in texts:
        for attempt in range(3):
            try:
                r = client.chat.completions.create(model=JUDGE_API["JUDGE_API_MODEL"], temperature=0,
                                                   max_tokens=CFG["judge_max_new"],
                                                   messages=[{"role": "system", "content": JUDGE_SYSTEM},
                                                             {"role": "user", "content": t}])
                outs.append(r.choices[0].message.content or "")
                break
            except Exception as e:
                if attempt == 2:
                    outs.append(f"ERROR: {e}")
                time.sleep(2 * (attempt + 1))
    return outs

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מונה ניסיונות חוזרים לפסקים לא קריאים.</p>
</div>

In [ ]:
JUDGE_RETRIES = {"asked": 0, "recovered": 0}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>_judge_call: מפנה לשופט המקומי או ל-API.</p>
</div>

In [ ]:
def _judge_call(prompts, jn):
    return judge_api(prompts) if jn.startswith("api:") else judge_local(prompts, jn)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>judge_texts: שולחת לשופט, עם מטמון וניסיון חוזר לפסק לא קריא.</p>
</div>

In [ ]:
def judge_texts(texts, name=None):
    jn = judge_name(name)
    todo = list({t for t in texts if (jn, t) not in JUDGE_CACHE})
    if todo:
        res = _judge_call(todo, jn)
        bad = [(t, r) for t, r in zip(todo, res) if parse_verdict(r)[0] == "invalid"]
        if bad:
            JUDGE_RETRIES["asked"] += len(bad)
            again = _judge_call([t + RETRY_SUFFIX for t, _ in bad], jn)
            fixed = {t: r for (t, _), r in zip(bad, again) if parse_verdict(r)[0] != "invalid"}
            JUDGE_RETRIES["recovered"] += len(fixed)
            res = [fixed.get(t, r) for t, r in zip(todo, res)]
        for t, r in zip(todo, res):
            JUDGE_CACHE[(jn, t)] = r
    return [JUDGE_CACHE[(jn, t)] for t in texts]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>parse_verdict: מחלצת פסק ונימוק.</p>
</div>

In [ ]:
def parse_verdict(text):
    m = re.search(r'"verdict"\s*:\s*"?\s*(A|B|tie)\b', text, re.I)
    if not m:
        m = re.search(r"\bverdict\b[^A-Za-z]{0,10}(A|B|tie)\b", text, re.I)
    v = m.group(1) if m else "invalid"
    v = v.lower() if v.lower() in ("tie", "invalid") else v.upper()
    r = re.search(r'"rationale"\s*:\s*"(.*?)"\s*,', text, re.S)
    return v, (r.group(1).strip() if r else text[:300])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>aggregate: מאחדת את שני הסדרים. פסק שמתהפך הוא תיקו.</p>
</div>

In [ ]:
def aggregate(v1, v2):
    m1 = {"A": "x", "B": "y", "tie": "tie", "invalid": "invalid"}[v1]
    m2 = {"A": "y", "B": "x", "tie": "tie", "invalid": "invalid"}[v2]
    if "invalid" in (m1, m2):
        return "tie", False, True
    if m1 == m2:
        return m1, False, False
    return "tie", (m1 != "tie" and m2 != "tie"), False

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>cut: מקצרת תשובה ל-1,500 תווים לפני השופט.</p>
</div>

In [ ]:
def cut(t):
    return t if len(t) <= CFG["judge_max_chars"] else t[:CFG["judge_max_chars"]] + " [...]"

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>compare: השוואה זוגית מלאה. מחזירה:</p>
<ul>
<li>win, tie, loss ו-win-rate עם רווח סמך, כנדרש</li>
<li>pref_decisive — ניצחונות מתוך ההשוואות שהוכרעו. זה המדד שמראה כיוון</li>
<li>שיעור היפוכים, פסקים לא קריאים ונטייה למשבצת הראשונה</li>
</ul>
</div>

In [ ]:
def compare(label, instructions, xs, ys, x_name, y_name, name=None):
    prompts = []
    for ins, x, y in zip(instructions, xs, ys):
        prompts += [JUDGE_TEMPLATE.format(instruction=ins, a=cut(x), b=cut(y)),
                    JUDGE_TEMPLATE.format(instruction=ins, a=cut(y), b=cut(x))]
    raw = judge_texts(prompts, name)
    rows, log = [], []
    for i, (ins, x, y) in enumerate(zip(instructions, xs, ys)):
        (v1, r1), (v2, r2) = parse_verdict(raw[2 * i]), parse_verdict(raw[2 * i + 1])
        res, flipped, invalid = aggregate(v1, v2)
        rows.append(dict(i=i, v_xy=v1, v_yx=v2, result=res, flipped=flipped, invalid=invalid,
                         len_x=len(words(x)), len_y=len(words(y)), rationale_xy=r1[:200], rationale_yx=r2[:200]))
        log += [dict(i=i, order="xy", raw=raw[2 * i][:700]), dict(i=i, order="yx", raw=raw[2 * i + 1][:700])]
    d = pd.DataFrame(rows)
    n = len(d)
    win, tie, loss = int((d.result == "x").sum()), int((d.result == "tie").sum()), int((d.result == "y").sum())
    first_slot = [v for v in list(d.v_xy) + list(d.v_yx) if v in ("A", "B")]
    summ = dict(label=label, x=x_name, y=y_name, judge=judge_name(name), n=n, win=win, tie=tie, loss=loss,
                win_rate=wilson(win, n), tie_rate=round(tie / n, 3), loss_rate=round(loss / n, 3),
                flip_rate=round(float(d.flipped.mean()), 3), invalid_rate=round(float(d.invalid.mean()), 3),
                p_first_slot=wilson(sum(v == "A" for v in first_slot), len(first_slot)),
                pref_decisive=(wilson(win, win + loss) if win + loss else (None, None, None)))
    RESULTS["comparisons"][label] = dict(summary=summ, rows=rows)
    RESULTS["judge_log"][label] = log
    print(f"{label}: {x_name} vs {y_name} | judge={summ['judge']} | n={n} | win {win} tie {tie} loss {loss} | "
          f"win-rate {summ['win_rate']} | decisive pref {summ['pref_decisive']} | flip {summ['flip_rate']} | invalid {summ['invalid_rate']}")
    return d, summ

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>checkpoint: שומרת את התוצאות לדיסק בסוף כל חלק.</p>
</div>

In [ ]:
def checkpoint(tag):
    path = os.path.join(RUN_DIR, "hw2_results.json")
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(RESULTS, f, indent=1, ensure_ascii=False, default=str)
        print(f"[checkpoint after {tag}] {path} ({os.path.getsize(path) / 1e6:.2f} MB)")
    except Exception as e:
        print(f"[checkpoint after {tag}] failed: {type(e).__name__}: {e}")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>summary_table: טבלת סיכום לכמה השוואות.</p>
</div>

In [ ]:
def summary_table(labels):
    rows = [RESULTS["comparisons"][l]["summary"] for l in labels if l in RESULTS["comparisons"]]
    return pd.DataFrame(rows)[["label", "x", "y", "n", "win", "tie", "loss", "win_rate", "tie_rate", "pref_decisive",
                               "flip_rate", "invalid_rate", "judge"]]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הדפסת תבנית השופט.</p>
</div>

In [ ]:
print(JUDGE_TEMPLATE)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון, זהה לכל השיטות פרט להבדלים האלה:</p>
<ul>
<li>full — כל המשקלים, float32 עם autocast ל-bf16, lr 2e-5</li>
<li>lora — r=16, alpha=32, כל השכבות הליניאריות, lr 2e-4</li>
<li>qlora — אותו LoRA מעל בסיס ב-NF4</li>
<li>ב-LoRA מתאמנת גם שורת im_end. בלעדיה, בהרצה מקדימה, אף תשובה של LoRA לא הסתיימה</li>
</ul>
<p>משותף לכולן: 2 epochs, אורך 512, batch 16, cosine, AdamW, לוס על התשובה בלבד.</p>
</div>

In [ ]:
LORA_ALL  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_ATTN = ["q_proj", "k_proj", "v_proj", "o_proj"]
CAN_QLORA = DEVICE == "cuda" and HAS_BNB
N_TOTAL_BASE = None

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>run_sft: מאמנת, ומודדת פרמטרים, זיכרון שיא, זמן ולוס.</p>
</div>

In [ ]:
def run_sft(name, method, train_ds, eval_ds, targets=None, epochs=None, lr=None):
    free_judge()
    free_gpu()
    method_eff = method
    if method == "qlora" and not CAN_QLORA:
        assert SMOKE, "QLoRA needs a CUDA GPU with bitsandbytes"
        print("[SMOKE] QLoRA is unavailable on CPU: running plain LoRA only to exercise the pipeline")
        method_eff = "lora"
    if method_eff == "full":
        model = load_base(dtype=torch.float32)
    elif method_eff == "lora":
        model = load_base()
    else:
        model = load_base(quant="4bit")
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    peft_cfg = None
    targets = targets or LORA_ALL
    if method_eff != "full":
        peft_cfg = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"], lora_dropout=CFG["lora_dropout"],
                              target_modules=targets, task_type="CAUSAL_LM",
                              trainable_token_indices={"embed_tokens": [IM_END]})
    epochs = epochs or CFG["epochs"]
    lr = lr or (CFG["lr_full"] if method_eff == "full" else CFG["lr_lora"])
    total_steps = math.ceil(len(train_ds) / (CFG["bsz"] * CFG["grad_acc"])) * epochs
    args = SFTConfig(
        output_dir=os.path.join(RUN_DIR, name), num_train_epochs=epochs,
        per_device_train_batch_size=CFG["bsz"], per_device_eval_batch_size=CFG["bsz"],
        gradient_accumulation_steps=CFG["grad_acc"], learning_rate=lr, lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.05 * total_steps)), weight_decay=0.0, max_grad_norm=1.0, optim="adamw_torch",
        bf16=BF16, fp16=(DEVICE == "cuda" and not BF16), max_length=CFG["max_len"], completion_only_loss=True, loss_type="nll",
        packing=False, logging_steps=max(1, total_steps // 20), logging_first_step=True, eval_strategy="epoch",
        save_strategy="no", report_to="none", seed=GLOBAL_SEED, dataloader_pin_memory=False,
        use_cpu=(DEVICE == "cpu"), disable_tqdm=False,
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                         processing_class=tok, peft_config=peft_cfg)
    n_tr, n_tot = count_params(trainer.model)
    free_gpu()
    out = trainer.train()
    peak = gib(torch.cuda.max_memory_allocated()) if DEVICE == "cuda" else None
    hist = trainer.state.log_history
    train_curve = [(h["step"], round(h["loss"], 4)) for h in hist if "loss" in h]
    eval_curve = [(round(h["epoch"], 2), round(h["eval_loss"], 4)) for h in hist if "eval_loss" in h]
    dtype_desc = {"full": f"float32 master weights, {ENV['dtype']} autocast",
                  "lora": f"{ENV['dtype']} frozen base, float32 LoRA adapters",
                  "qlora": f"NF4 double-quant frozen base, {ENV['dtype']} compute, float32 LoRA adapters"}[method_eff]
    RESULTS["runs"][name] = dict(
        method=method, method_effective=method_eff, targets=(targets if method_eff != "full" else "all weights"),
        n_examples=len(train_ds), epochs=epochs, steps=total_steps, lr=lr, effective_batch=CFG["bsz"] * CFG["grad_acc"],
        max_len=CFG["max_len"], dtype=dtype_desc, seed=GLOBAL_SEED, trainable=n_tr, total_reported=n_tot,
        trainable_pct=round(100 * n_tr / N_TOTAL_BASE, 4), peak_train_gib=peak,
        train_runtime_s=round(out.metrics["train_runtime"], 1), train_loss_final=(train_curve[-1][1] if train_curve else None),
        train_curve=train_curve, eval_losses=eval_curve,
    )
    r = RESULTS["runs"][name]
    print(f"{name}: {method_eff} | trainable {n_tr:,} ({r['trainable_pct']}%) | steps {total_steps} | "
          f"time {r['train_runtime_s']}s | peak {peak} GiB | final train loss {r['train_loss_final']} | eval {eval_curve}")
    return trainer.model

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>load_sft: טוענת את מודל ה-SFT השמור, לחלקים ד׳ עד ו׳.</p>
</div>

In [ ]:
def load_sft(path):
    kw = {"dtype": DTYPE}
    if DEVICE == "cuda":
        kw["device_map"] = {"": 0}
    m = AutoModelForCausalLM.from_pretrained(path, **kw)
    m.config.pad_token_id = tok.pad_token_id
    return m

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הדאטה, Dolly-15k:</p>
<ul>
<li>סינון תשובות קצרות או ארוכות מדי, והגרלה לפי seed</li>
<li>3,000 לאימון, 200 ל-held-out</li>
<li>מהיתר, בלי חפיפה: 300 לחלק ד׳, 70 לגיבוי, 50 לסט השופט</li>
</ul>
</div>

In [ ]:
raw = load_dataset(DATASET, split="train")
df = raw.to_pandas()
df["context"] = df["context"].fillna("")
ok = (df.response.str.len().between(20, 1500) & df.instruction.str.len().between(10, 400) & (df.context.str.len() <= 1200))
df = df[ok].sample(frac=1.0, random_state=GLOBAL_SEED).reset_index(drop=True)
n_tr, n_ev = CFG["n_train"], CFG["n_eval"]
train_df, eval_df = df.iloc[:n_tr].copy(), df.iloc[n_tr:n_tr + n_ev].copy()
pool_df = df.iloc[n_tr + n_ev:]
pool_df = pool_df[pool_df.context.str.len() == 0].sample(frac=1.0, random_state=GLOBAL_SEED + 1).reset_index(drop=True)
a, b = CFG["n_pref_prompts"], CFG["n_pref_prompts"] + CFG["n_pref_extra"]
pref_df, pref_extra_df, judge_df = pool_df.iloc[:a].copy(), pool_df.iloc[a:b].copy(), pool_df.iloc[b:b + CFG["n_judge"]].copy()
for d in (pref_df, pref_extra_df, judge_df):
    d.reset_index(drop=True, inplace=True)
JUDGE_PROMPTS = [to_prompt(i) for i in judge_df.instruction]
JUDGE_INSTR = judge_df.instruction.tolist()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>to_sft: שורת Dolly לזוג prompt-completion, עם im_end בסוף.</p>
</div>

In [ ]:
def to_sft(d):
    return Dataset.from_dict({"prompt": [to_prompt(i, c) for i, c in zip(d.instruction, d.context)],
                              "completion": [r.strip() + "<|im_end|>" for r in d.response]})

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בניית הסטים: אימון, held-out, 200 לחלק ג׳, והסט הרועש.</p>
</div>

In [ ]:
train_ds, eval_ds = to_sft(train_df), to_sft(eval_df)
small_ds = to_sft(train_df.iloc[:CFG["n_small"]])

rng = np.random.RandomState(GLOBAL_SEED)
noisy_df = train_df.copy()
noise_idx = rng.choice(len(noisy_df), size=int(CFG["noise_rate"] * len(noisy_df)), replace=False)
shifted = np.roll(noisy_df.response.values, 1)
noisy_df.iloc[noise_idx, noisy_df.columns.get_loc("response")] = shifted[noise_idx]
noisy_ds = to_sft(noisy_df)

lens = [len(tok(p + c)["input_ids"]) for p, c in zip(train_ds["prompt"][:300], train_ds["completion"][:300])]
RESULTS["data"] = dict(dataset=DATASET, rows_total=len(raw), rows_after_filter=len(df), n_train=len(train_ds),
                       n_eval=len(eval_ds), n_small=len(small_ds), n_noisy=len(noisy_ds), noise_rate=CFG["noise_rate"],
                       n_noised=int(len(noise_idx)), pool_no_context=len(pool_df), n_pref=len(pref_df), n_pref_extra=len(pref_extra_df), n_judge=len(judge_df),
                       token_len_mean=round(float(np.mean(lens)), 1), token_len_p95=float(np.percentile(lens, 95)),
                       frac_over_max_len=round(float(np.mean([l > CFG["max_len"] for l in lens])), 3))
print(json.dumps(RESULTS["data"], indent=1))
display(train_df.category.value_counts().rename("train examples per category").to_frame())
print("example:", repr(train_ds[0]["prompt"][-80:]), "->", repr(train_ds[0]["completion"][:80]))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרף הדאטה: אורך הדוגמאות והקטגוריות.</p>
</div>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].hist(lens, bins=30, color="tab:blue")
ax[0].axvline(CFG["max_len"], color="k", ls="--", lw=1, label=f"max_len = {CFG['max_len']}")
ax[0].set_xlabel("tokens per example, prompt and answer"); ax[0].set_ylabel("examples")
ax[0].set_title(f"length of {len(lens)} training examples"); ax[0].legend(fontsize=8)
cats = train_df.category.value_counts()
ax[1].barh(cats.index, cats.values, color="tab:gray"); ax[1].invert_yaxis()
ax[1].set_title("training examples per category"); ax[1].set_xlabel("examples"); ax[1].set_ylabel("Dolly category")
plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הסטטיסטיקה של אורכי הטוקנים מוצגת כדי לוודא ששיעור זניח של דוגמאות עובר את 512 ונחתך. חלוקת הקטגוריות מראה שהסט מגוון: שאלות פתוחות, סיעור מוחות, סיווג, סיכום ושאלות עם הקשר, ולכן שינוי ההתנהגות שיימדד אינו של סוג משימה יחיד.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>פרוטוקול ההערכה, אחיד לכל מודל:</p>
<ul>
<li>greedy על 12 הפרומפטים, למדדים האוטומטיים</li>
<li>שתי דגימות בטמפרטורה 0.7, בשני seeds, על הקבועים ועל 50 פרומפטי השופט</li>
<li>compare_fixed — 24 השוואות, הסט שסעיף 7 דורש. compare_judgeset — 100 השוואות</li>
<li>מול הבסיס רצות שתיהן. בהרצות קודמות 24 השוואות לא הספיקו להכריע</li>
</ul>
</div>

In [ ]:
resp, SAMP, JSAMP = {}, {}, {}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>gen_eval: מייצרת את כל הפלטים של מודל ושומרת אותם לפי שמו.</p>
</div>

In [ ]:
def gen_eval(model, name):
    resp[name], _ = generate(model, FIXED_PROMPTS)
    for s in CFG["eval_seeds"]:
        SAMP[(name, s)], _ = generate(model, FIXED_PROMPTS, temperature=CFG["eval_temp"], seed=GLOBAL_SEED + s)
        JSAMP[(name, s)], _ = generate(model, JUDGE_PROMPTS, temperature=CFG["eval_temp"], seed=GLOBAL_SEED + s)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>texts: מאחדת את תשובות שני ה-seeds לרשימה אחת.</p>
</div>

In [ ]:
def texts(store, name):
    return sum(([o["text"] for o in store[(name, s)]] for s in CFG["eval_seeds"]), [])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>compare_fixed: 12 הפרומפטים הקבועים, 24 השוואות. זה הסט שסעיף 7 דורש.</p>
</div>

In [ ]:
def compare_fixed(label, x, y, name=None):
    return compare(label, INSTR * len(CFG["eval_seeds"]), texts(SAMP, x), texts(SAMP, y), x, y, name)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>compare_judgeset: 50 פרומפטי השופט, 100 השוואות.</p>
</div>

In [ ]:
def compare_judgeset(label, x, y, name=None):
    return compare(label, JUDGE_INSTR * len(CFG["eval_seeds"]), texts(JSAMP, x), texts(JSAMP, y), x, y, name)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הדפסת גודל הסטים.</p>
</div>

In [ ]:
print(f"fixed set: {len(FIXED_PROMPTS)} prompts x {len(CFG['eval_seeds'])} seeds | judge set: {len(JUDGE_PROMPTS)} prompts x {len(CFG['eval_seeds'])} seeds")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>פונקציות עזר לגרפים.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>run_facts: הרשומה של אימון אחד.</p>
</div>

In [ ]:
def run_facts(name):
    return RESULTS["runs"].get(name, {})

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>plot_outcomes: פס של ניצחונות, תיקו והפסדים לכל השוואה.</p>
</div>

In [ ]:
def plot_outcomes(labels, title="judge outcomes"):
    ss = [RESULTS["comparisons"][l]["summary"] for l in labels if l in RESULTS["comparisons"]]
    if not ss:
        return
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(ss) + 1.4))
    y = np.arange(len(ss))
    left = np.zeros(len(ss))
    for key, color, lab in (("win", "tab:green", "first model wins"), ("tie", "lightgray", "tie"), ("loss", "tab:red", "second model wins")):
        v = np.array([s[key] / s["n"] for s in ss])
        ax.barh(y, v, left=left, color=color, label=lab)
        for yi, (l0, w, s) in enumerate(zip(left, v, ss)):
            if w > 0.05:
                ax.text(l0 + w / 2, yi, str(s[key]), ha="center", va="center", fontsize=8)
        left += v
    names = [f"{s['x']} vs {s['y']} (n={s['n']})" + (" [judge 2]" if s["judge"] != judge_name() else "") for s in ss]
    ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.set_xlabel("share of comparisons"); ax.set_ylabel("comparison"); ax.set_title(title)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), fontsize=8, ncol=3, frameon=False)
    plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>plot_behavior: ארבעת המדדים האוטומטיים לכמה מודלים.</p>
</div>

In [ ]:
def plot_behavior(stats, title="automatic metrics"):
    cols = [("mean_words", "mean words"), ("ended_rate", "ended by itself"), ("rep4", "4-gram repetition"), ("check_pass", "checks passed")]
    units = {"mean_words": "words per answer", "ended_rate": "share of answers", "rep4": "share of 4-grams", "check_pass": "share of prompts"}
    cols = [(c, t) for c, t in cols if c in stats.columns]
    fig, ax = plt.subplots(1, len(cols), figsize=(3.1 * len(cols), 3.3))
    colors = plt.cm.tab10(np.arange(len(stats)))
    for a, (c, t) in zip(np.atleast_1d(ax), cols):
        a.bar(stats.index.astype(str), stats[c], color=colors)
        a.set_title(t, fontsize=10)
        a.set_ylabel(units[c], fontsize=9)
        a.set_xlabel("model", fontsize=9)
        a.tick_params(axis="x", rotation=35, labelsize=8)
        if c != "mean_words":
            a.set_ylim(0, 1.05)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>plot_losses: לוס אימון ולוס held-out לכמה אימונים.</p>
</div>

In [ ]:
def plot_losses(names, title="training curves"):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for n in names:
        r = run_facts(n)
        tc, ec = r.get("train_curve") or [], r.get("eval_losses") or []
        if tc:
            steps = np.array([st for st, _ in tc], dtype=float)
            ax[0].plot(steps / max(r.get("steps") or steps.max(), 1), [l for _, l in tc], label=n)
        if ec:
            ax[1].plot([e for e, _ in ec], [l for _, l in ec], marker="o", label=n)
    ax[0].set_xlabel("fraction of training"); ax[0].set_ylabel("train loss"); ax[0].set_title("train loss, completion tokens only")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("held-out loss"); ax[1].set_title("held-out loss at the end of each epoch")
    for a in ax:
        a.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p></p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק א׳ — SFT בסיסי</p>
<ul>
<li>סעיפים 1–4: Qwen2.5-0.5B base, 3,000 דוגמאות, Full FT, 12 פרומפטים לפני ואחרי</li>
<li>נמדד: אורך, סיום, חזרתיות, בדיקות, לוס held-out ושינון</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תשובות הבסיס, לפני אימון.</p>
</div>

In [ ]:
base = load_base()
N_TOTAL_BASE = sum(p.numel() for p in base.parameters())
gen_eval(base, "base")
print(f"base model: {N_TOTAL_BASE:,} parameters")
for f, o in zip(FIXED, resp["base"]):
    print(f"--- {f['id']} | {o['n_tokens']} tokens | ended={o['ended']}\n{o['text'][:350]}\n")
base = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון Full FT ושמירת המודל לדיסק.</p>
</div>

In [ ]:
sft_model = run_sft("full_ft", "full", train_ds, eval_ds)
SFT_DIR = os.path.join(RUN_DIR, "sft_full")
sft_model = sft_model.to(DTYPE)
sft_model.save_pretrained(SFT_DIR)
tok.save_pretrained(SFT_DIR)
print("saved to", SFT_DIR, "| dtype", DTYPE)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תשובות אחרי ה-SFT, לצד התשובות שלפני.</p>
</div>

In [ ]:
gen_eval(sft_model, "full_ft")
side = pd.DataFrame({"id": [f["id"] for f in FIXED],
                     "before (base)": [o["text"][:110].replace("\n", " ") for o in resp["base"]],
                     "after (full FT)": [o["text"][:110].replace("\n", " ") for o in resp["full_ft"]]})
display(side)
for f, o in zip(FIXED, resp["full_ft"]):
    print(f"--- {f['id']} | {o['n_tokens']} tokens | ended={o['ended']}\n{o['text'][:350]}\n")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>טבלת המדדים לפני ואחרי, ובדיקות לפי פרומפט.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>check_table: pass או fail לכל פרומפט ומודל.</p>
</div>

In [ ]:
def check_table(keys):
    return pd.DataFrame({k: ["pass" if f["check"](o["text"]) else "fail" for f, o in zip(FIXED, resp[k])] for k in keys},
                        index=[f"{f['id']} ({f['cat']})" for f in FIXED])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חישוב הטבלאות ושמירתן.</p>
</div>

In [ ]:
stats_a = pd.DataFrame([behavior_stats(resp["base"]), behavior_stats(resp["full_ft"])], index=["base", "full_ft"])
display(stats_a)
display(check_table(["base", "full_ft"]))
RESULTS["part_a"] = {"behavior": stats_a.to_dict("index"), "checks": check_table(["base", "full_ft"]).to_dict()}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אותם מדדים בגרף.</p>
</div>

In [ ]:
plot_behavior(stats_a, "Part A: base vs full fine-tuning, 12 fixed prompts, greedy")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>עקומות הלוס של Full FT.</p>
</div>

In [ ]:
r = RESULTS["runs"]["full_ft"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
if r["train_curve"]:
    ax[0].plot([s for s, _ in r["train_curve"]], [l for _, l in r["train_curve"]], marker=".")
ax[0].set_title("Full FT: training loss (completion tokens)"); ax[0].set_xlabel("optimizer step"); ax[0].set_ylabel("loss")
if r["eval_losses"]:
    ax[1].plot([e for e, _ in r["eval_losses"]], [l for _, l in r["eval_losses"]], marker="o")
ax[1].set_title("Full FT: held-out loss per epoch"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("eval loss")
plt.tight_layout(); plt.show()
print("eval losses per epoch:", r["eval_losses"])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מבחן שינון: 8 הוראות מהאימון ו-8 שהמודל לא ראה, מול התשובה המקורית בדאטה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>memorization: lcs_ratio לכל תשובה מול תשובת הדאטה.</p>
</div>

In [ ]:
def memorization(model, d):
    outs, _ = generate(model, [to_prompt(i, c) for i, c in zip(d.instruction, d.context)])
    return [lcs_ratio(o["text"], r) for o, r in zip(outs, d.response)], outs

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הרצת המבחן והצגת הדוגמה ששוחזרה הכי טוב.</p>
</div>

In [ ]:
n_probe = 8 if not SMOKE else 3
mem_tr, out_tr = memorization(sft_model, train_df.iloc[:n_probe])
mem_ev, out_ev = memorization(sft_model, eval_df.iloc[:n_probe])
mem = pd.DataFrame({"mean lcs_ratio": [np.mean(mem_tr), np.mean(mem_ev)], "max": [max(mem_tr), max(mem_ev)],
                    "n >= 0.8": [sum(m >= 0.8 for m in mem_tr), sum(m >= 0.8 for m in mem_ev)]},
                   index=["train prompts (seen)", "held-out prompts (unseen)"]).round(3)
display(mem)
RESULTS["part_a"]["memorization"] = dict(train=mem_tr, heldout=mem_ev)
i_max = int(np.argmax(mem_tr))
print("most reproduced training example, lcs_ratio", mem_tr[i_max])
print("instruction:", train_df.instruction.iloc[i_max][:200])
print("original   :", train_df.response.iloc[i_max][:250])
print("generated  :", out_tr[i_max]["text"][:250])
sft_model = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>המבחן בגרף: נקודה לכל הוראה, קו שחור לממוצע, קו אדום ל-0.8.</p>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.3))
for k, vals in enumerate([mem_tr, mem_ev]):
    jitter = np.random.RandomState(k).uniform(-0.08, 0.08, len(vals))
    ax.scatter(np.full(len(vals), k) + jitter, vals, s=40, alpha=0.8)
    ax.hlines(np.mean(vals), k - 0.25, k + 0.25, color="k")
ax.axhline(0.8, color="tab:red", ls="--", lw=1)
ax.set_xticks([0, 1]); ax.set_xticklabels(["train prompts (seen)", "held-out prompts (unseen)"])
ax.set_ylim(0, 1.05); ax.set_ylabel("lcs_ratio vs dataset answer"); ax.set_title("memorization probe, one dot per prompt")
plt.tight_layout(); plt.show()
checkpoint("חלק א׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק א׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק ב׳ — Full FT מול LoRA מול QLoRA</p>
<ul>
<li>סעיפים 5–7: אותו אימון בשלוש שיטות. פרמטרים, זיכרון שיא, זמן, ושופט מול הבסיס</li>
<li>כל שיטה מול הבסיס פעמיים: 24 השוואות בסט הקבוע ו-100 בסט השופט</li>
<li>בנוסף, על 100 השוואות: QLoRA מול LoRA, ו-LoRA מול Full FT</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון LoRA.</p>
</div>

In [ ]:
m = run_sft("lora", "lora", train_ds, eval_ds)
gen_eval(m, "lora")
m = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון QLoRA.</p>
</div>

In [ ]:
m = run_sft("qlora", "qlora", train_ds, eval_ds)
gen_eval(m, "qlora")
m = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>טבלת המשאבים וגרף.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>run_table: טבלת משאבים ולוס לכמה אימונים.</p>
</div>

In [ ]:
def run_table(names):
    rows = []
    for n in names:
        r = RESULTS["runs"][n]
        rows.append(dict(run=n, method=r["method_effective"], n_examples=r["n_examples"], steps=r["steps"], lr=r["lr"],
                         trainable=r["trainable"], trainable_pct=r["trainable_pct"], peak_gib=r["peak_train_gib"],
                         time_s=r["train_runtime_s"], train_loss=r["train_loss_final"],
                         eval_loss=(r["eval_losses"][-1][1] if r["eval_losses"] else None), dtype=r["dtype"]))
    return pd.DataFrame(rows).set_index("run")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חישוב הטבלה והגרף.</p>
</div>

In [ ]:
tab_b = run_table(["full_ft", "lora", "qlora"])
display(tab_b)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].bar(tab_b.index, tab_b.trainable_pct); ax[0].set_title("trainable parameters"); ax[0].set_yscale("log")
ax[0].set_ylabel("% of all parameters, log scale")
ax[1].bar(tab_b.index, tab_b.peak_gib.fillna(0)); ax[1].set_title("peak training memory"); ax[1].set_ylabel("GiB")
ax[2].bar(tab_b.index, tab_b.time_s); ax[2].set_title("training time"); ax[2].set_ylabel("seconds")
for a in ax:
    a.set_xlabel("method")
plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>השופט: כל שיטה מול הבסיס על שני הסטים, ושתי השוואות צמודות על סט השופט.</p>
</div>

In [ ]:
X = lambda k: [o["text"] for o in resp[k]]
compare_fixed("B_full_vs_base", "full_ft", "base")
compare_fixed("B_lora_vs_base", "lora", "base")
compare_fixed("B_qlora_vs_base", "qlora", "base")
compare_judgeset("B_full_vs_base_js", "full_ft", "base")
compare_judgeset("B_lora_vs_base_js", "lora", "base")
compare_judgeset("B_qlora_vs_base_js", "qlora", "base")
compare_judgeset("B_qlora_vs_lora", "qlora", "lora")
compare_judgeset("B_lora_vs_full", "lora", "full_ft")
display(summary_table(["B_full_vs_base", "B_lora_vs_base", "B_qlora_vs_base", "B_full_vs_base_js", "B_lora_vs_base_js",
                       "B_qlora_vs_base_js", "B_qlora_vs_lora", "B_lora_vs_full"]))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מדדים אוטומטיים לארבעת המודלים.</p>
</div>

In [ ]:
stats_b = pd.DataFrame([behavior_stats(resp[k]) for k in ["base", "full_ft", "lora", "qlora"]],
                       index=["base", "full_ft", "lora", "qlora"])
display(stats_b)
display(check_table(["base", "full_ft", "lora", "qlora"]))
RESULTS["part_b"] = {"resources": tab_b.to_dict("index"), "behavior": stats_b.to_dict("index")}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרפים לחלק ב׳: לוס, מדדים אוטומטיים, תוצאות השופט.</p>
</div>

In [ ]:
plot_losses(["full_ft", "lora", "qlora"], "Part B: the three fine-tuning methods")
plot_behavior(stats_b, "Part B: automatic metrics, 12 fixed prompts, greedy")
plot_outcomes(["B_full_vs_base", "B_lora_vs_base", "B_qlora_vs_base", "B_full_vs_base_js", "B_lora_vs_base_js",
               "B_qlora_vs_base_js", "B_qlora_vs_lora", "B_lora_vs_full"], "Part B: judge outcomes")
checkpoint("חלק ב׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק ב׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק ג׳ — גודל ואיכות הדאטה</p>
<ul>
<li>סעיפים 8–9: LoRA על 200 דוגמאות, ועל 3,000 שב-30% מהן התשובה הוחלפה</li>
<li>כל תצורה מול הבסיס, ומול LoRA על הדאטה המלא</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון שתי התצורות.</p>
</div>

In [ ]:
m = run_sft("lora_small", "lora", small_ds, eval_ds)
gen_eval(m, "lora_small")
m = None
free_gpu()
m = run_sft("lora_noisy", "lora", noisy_ds, eval_ds)
gen_eval(m, "lora_noisy")
m = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>טבלאות, מדדים ושופט לשלוש התצורות.</p>
</div>

In [ ]:
tab_c = run_table(["lora", "lora_small", "lora_noisy"])
display(tab_c[["n_examples", "steps", "time_s", "train_loss", "eval_loss"]])
stats_c = pd.DataFrame([behavior_stats(resp[k]) for k in ["base", "lora", "lora_small", "lora_noisy"]],
                       index=["base", "lora", "lora_small", "lora_noisy"])
display(stats_c)
display(check_table(["lora", "lora_small", "lora_noisy"]))
compare_fixed("C_small_vs_base", "lora_small", "base")
compare_fixed("C_noisy_vs_base", "lora_noisy", "base")
compare_judgeset("C_small_vs_lora", "lora_small", "lora")
compare_judgeset("C_noisy_vs_lora", "lora_noisy", "lora")
display(summary_table(["C_small_vs_base", "C_noisy_vs_base", "C_small_vs_lora", "C_noisy_vs_lora"]))
RESULTS["part_c"] = {"runs": tab_c.to_dict("index"), "behavior": stats_c.to_dict("index")}

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרפים לחלק ג׳.</p>
</div>

In [ ]:
plot_losses(["lora", "lora_small", "lora_noisy"], "Part C: data size and data quality")
plot_behavior(stats_c, "Part C: automatic metrics, 12 fixed prompts, greedy")
plot_outcomes(["C_small_vs_base", "C_noisy_vs_base", "C_small_vs_lora", "C_noisy_vs_lora"], "Part C: judge outcomes")
checkpoint("חלק ג׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק ג׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק ד׳ — דאטת העדפות ו-reward model</p>
<ul>
<li>סעיפים 10–13: 300 פרומפטים חדשים, שתי דגימות מה-SFT, שופט זוגי, Bradley-Terry מעל הטמעות</li>
<li>בהרצות קודמות, עם 80 פרומפטים, נשארו פחות מ-40 זוגות וה-reward model היה ברמת ניחוש</li>
<li>זוג שהתהפך או יצא תיקו לא נכנס לדאטה</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>דגימת שתי תשובות לכל פרומפט.</p>
</div>

In [ ]:
PREF_PROMPTS = [to_prompt(i) for i in pref_df.instruction]
sft = load_sft(SFT_DIR)
outs, t = generate(sft, PREF_PROMPTS, temperature=CFG["pref_temp"], top_p=CFG["pref_top_p"], seed=GLOBAL_SEED + 7, num_return=2)
samp1, samp2 = outs[0::2], outs[1::2]
print(f"{len(PREF_PROMPTS)} prompts x 2 samples in {t}s | mean words: {np.mean([len(words(o['text'])) for o in outs]):.1f}")
for k in range(2):
    print(f"\n=== prompt {k}: {pref_df.instruction[k][:120]}")
    print("sample 1:", samp1[k]["text"][:200].replace("\n", " "))
    print("sample 2:", samp2[k]["text"][:200].replace("\n", " "))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תיוג ההעדפות בידי השופט, ומדדי ההטיה.</p>
</div>

In [ ]:
d_pref, s_pref = compare("D_pref", pref_df.instruction.tolist(), [o["text"] for o in samp1], [o["text"] for o in samp2], "sample_1", "sample_2")
decisive = d_pref[d_pref.result.isin(["x", "y"])].copy()
decisive["longer_won"] = np.where(decisive.result == "x", decisive.len_x > decisive.len_y, decisive.len_y > decisive.len_x)
decisive["same_len"] = decisive.len_x == decisive.len_y
lw = decisive[~decisive.same_len]
bias = dict(n_pairs=len(d_pref), n_decisive=int(len(decisive)), tie_rate=s_pref["tie_rate"], flip_rate=s_pref["flip_rate"],
            invalid_rate=s_pref["invalid_rate"], p_first_slot=s_pref["p_first_slot"],
            p_longer_wins=wilson(int(lw.longer_won.sum()), len(lw)))
print(json.dumps(bias, indent=1))
RESULTS.setdefault("part_d", {})["judge_bias"] = bias
borderline = d_pref[(d_pref.result == "tie")].head(3)
for _, row in borderline.iterrows():
    print(f"\n--- borderline pair {row.i} | verdicts {row.v_xy}/{row.v_yx} | flipped={row.flipped} | words {row.len_x} vs {row.len_y}")
    print("instruction:", pref_df.instruction[row.i][:150])
    print("rationale (order xy):", row.rationale_xy[:200])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שתי ההטיות בגרף.</p>
</div>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
vals = [("picks the first slot", bias["p_first_slot"]), ("longer answer wins", bias["p_longer_wins"])]
for k, (lab, (p, lo, hi)) in enumerate(vals):
    ax[0].bar(k, p, color="tab:purple")
    ax[0].errorbar(k, p, yerr=[[p - lo], [hi - p]], color="k", capsize=5)
ax[0].axhline(0.5, color="k", ls="--", lw=1)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels([v[0] for v in vals]); ax[0].set_ylim(0, 1)
ax[0].set_title("judge biases with 95% Wilson CI, 0.5 = no bias"); ax[0].set_ylabel("probability")
for res, color, lab in (("x", "tab:green", "sample 1 chosen"), ("y", "tab:red", "sample 2 chosen"), ("tie", "lightgray", "tie")):
    part = d_pref[d_pref.result == res]
    ax[1].scatter(part.len_x, part.len_y, c=color, label=f"{lab} ({len(part)})", s=25, alpha=0.8)
lim = max(d_pref.len_x.max(), d_pref.len_y.max()) + 5
ax[1].plot([0, lim], [0, lim], color="k", lw=0.8)
ax[1].set_xlabel("words in sample 1"); ax[1].set_ylabel("words in sample 2")
ax[1].set_title("each pair by the length of its two samples"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בניית הזוגות וחלוקה ל-train ו-held-out. אם יש פחות מ-40 זוגות, רץ נוהל גיבוי של 70 פרומפטים.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>pairs_from: רק זוג שהוכרע נכנס, עם chosen ו-rejected.</p>
</div>

In [ ]:
def pairs_from(d, frame, s1, s2, prompts):
    out = []
    for _, row in d.iterrows():
        if row.result in ("x", "y"):
            c, r = (s1, s2) if row.result == "x" else (s2, s1)
            out.append(dict(i=int(row.i), instruction=frame.instruction[row.i], prompt=prompts[row.i],
                            chosen=c[row.i]["text"], rejected=r[row.i]["text"]))
    return out

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בניית הזוגות.</p>
</div>

In [ ]:
pairs = pairs_from(d_pref, pref_df, samp1, samp2, PREF_PROMPTS)
RESULTS["part_d"]["fallback_used"] = False
if len(pairs) < CFG["min_pairs"] and len(pref_extra_df):
    print(f"only {len(pairs)} decisive pairs (< {CFG['min_pairs']}): running the pre-registered fallback on {len(pref_extra_df)} extra prompts")
    EXTRA_PROMPTS = [to_prompt(i) for i in pref_extra_df.instruction]
    outs2, _ = generate(sft, EXTRA_PROMPTS, temperature=CFG["pref_temp"], top_p=CFG["pref_top_p"], seed=GLOBAL_SEED + 8, num_return=2)
    x1, x2 = outs2[0::2], outs2[1::2]
    d_extra, s_extra = compare("D_pref_extra", pref_extra_df.instruction.tolist(), [o["text"] for o in x1], [o["text"] for o in x2], "sample_1", "sample_2")
    extra_pairs = pairs_from(d_extra, pref_extra_df, x1, x2, EXTRA_PROMPTS)
    for p in extra_pairs:
        p["i"] += 1000
    pairs += extra_pairs
    RESULTS["part_d"].update(fallback_used=True, n_pairs_first_round=len(pairs) - len(extra_pairs), n_pairs_fallback=len(extra_pairs))
if SMOKE and len(pairs) < 4:
    print("[SMOKE] too few decisive pairs; using all pairs with an arbitrary orientation")
    pairs = [dict(i=k, instruction=pref_df.instruction[k], prompt=PREF_PROMPTS[k], chosen=samp1[k]["text"], rejected=samp2[k]["text"])
             for k in range(len(PREF_PROMPTS))]
rng = np.random.RandomState(GLOBAL_SEED)
order = rng.permutation(len(pairs))
n_hold = min(CFG["n_pref_holdout"], max(1, len(pairs) // 4))
pref_hold = [pairs[k] for k in order[:n_hold]]
pref_train = [pairs[k] for k in order[n_hold:]]
RESULTS["part_d"].update(n_pairs_decisive=len(pairs), n_pref_train=len(pref_train), n_pref_holdout=len(pref_hold))
print(f"decisive pairs {len(pairs)} -> train {len(pref_train)} | held-out {len(pref_hold)}")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הטמעות ל-reward model: ממוצע המצב החבוי האחרון של ה-SFT על טוקני התשובה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>embed_responses: וקטור של 896 ממדים לכל תשובה.</p>
</div>

In [ ]:
@torch.no_grad()
def embed_responses(model, prompts, responses, batch_size=8):
    model.eval()
    tok.padding_side = "right"
    feats = []
    for i in range(0, len(prompts), batch_size):
        ps, rs = prompts[i:i + batch_size], responses[i:i + batch_size]
        enc = tok([p + r + "<|im_end|>" for p, r in zip(ps, rs)], return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG["max_len"] + CFG["max_new"]).to(model.device)
        plen = [len(tok(p)["input_ids"]) for p in ps]
        h = model(**enc, output_hidden_states=True).hidden_states[-1].float()
        for j in range(len(ps)):
            mask = enc["attention_mask"][j].clone().float()
            mask[:plen[j]] = 0
            feats.append(((h[j] * mask[:, None]).sum(0) / mask.sum().clamp(min=1)).cpu().numpy())
    return np.stack(feats)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חישוב ההטמעות.</p>
</div>

In [ ]:
t0 = time.perf_counter()
E_c = embed_responses(sft, [p["prompt"] for p in pref_train], [p["chosen"] for p in pref_train])
E_r = embed_responses(sft, [p["prompt"] for p in pref_train], [p["rejected"] for p in pref_train])
H_c = embed_responses(sft, [p["prompt"] for p in pref_hold], [p["chosen"] for p in pref_hold])
H_r = embed_responses(sft, [p["prompt"] for p in pref_hold], [p["rejected"] for p in pref_hold])
print(f"embeddings {E_c.shape} in {time.perf_counter() - t0:.1f}s")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון ה-reward model: רגרסיה לוגיסטית על הפרש ההטמעות, בלי איבר חופשי.</p>
<ul>
<li>עוצמת הרגולריזציה C נבחרת מתוך 0.001, 0.01, 0.1, 1 ב-cross-validation פנימי לפי זוגות</li>
<li>בהרצות קודמות, עם C קבוע, המודל הגיע ל-100% באימון ולרמת ניחוש ב-cross-validation</li>
</ul>
</div>

In [ ]:
scaler = StandardScaler().fit(np.vstack([E_c, E_r]))
D = scaler.transform(E_c) - scaler.transform(E_r)
Xrm = np.vstack([D, -D])
yrm = np.r_[np.ones(len(D)), np.zeros(len(D))]
grp = np.r_[np.arange(len(D)), np.arange(len(D))]

RM_CS = [0.001, 0.01, 0.1, 1.0]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>fit_rm: רגרסיה לוגיסטית, עם C שנבחר ב-cross-validation פנימי לפי זוגות.</p>
</div>

In [ ]:
def fit_rm(X, y, groups):
    k = min(5, len(set(groups)))
    best_c, best_acc = 0.1, -1.0
    if k >= 2:
        for c in RM_CS:
            hits = []
            for tr, te in GroupKFold(n_splits=k).split(X, y, groups):
                m = LogisticRegression(C=c, fit_intercept=False, max_iter=5000).fit(X[tr], y[tr])
                hits += list(m.predict(X[te]) == y[te])
            if np.mean(hits) > best_acc:
                best_c, best_acc = c, float(np.mean(hits))
    return LogisticRegression(C=best_c, fit_intercept=False, max_iter=5000).fit(X, y), best_c

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הערכה ב-cross-validation לפי זוגות, ואימון סופי.</p>
</div>

In [ ]:
n_splits = min(5, len(D))
cv_hits = []
if n_splits >= 2:
    for tr_idx, te_idx in GroupKFold(n_splits=n_splits).split(Xrm, yrm, grp):
        clf, _ = fit_rm(Xrm[tr_idx], yrm[tr_idx], grp[tr_idx])
        te_pairs = sorted(set(grp[te_idx]))
        cv_hits += [int(clf.decision_function(D[[k]])[0] > 0) for k in te_pairs]
rm, rm_c = fit_rm(Xrm, yrm, grp)
w_rm = rm.coef_[0]

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>rm_score: ציון התגמול של תשובות.</p>
</div>

In [ ]:
def rm_score(E):
    return scaler.transform(E) @ w_rm

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מדדי ה-reward model.</p>
</div>

In [ ]:
hold_diff = rm_score(H_c) - rm_score(H_r)
rm_stats = dict(n_train_pairs=len(D), n_features=int(D.shape[1]),
                cv_accuracy=wilson(sum(cv_hits), len(cv_hits)), holdout_accuracy=wilson(int((hold_diff > 0).sum()), len(hold_diff)),
                train_accuracy=round(float((D @ w_rm > 0).mean()), 3), chosen_C=rm_c)
print(json.dumps(rm_stats, indent=1))
RESULTS["part_d"]["reward_model"] = rm_stats

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרף סעיף 13: הפרש ה-reward לכל זוג.</p>
</div>

In [ ]:
n_show = min(20, len(D) + len(hold_diff))
train_diff = D @ w_rm
items = [(float(v), "held-out") for v in hold_diff] + [(float(v), "train") for v in train_diff[:max(0, n_show - len(hold_diff))]]
items = sorted(items, key=lambda x: x[0])
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(range(len(items)), [v for v, _ in items], color=["tab:orange" if s == "held-out" else "tab:blue" for _, s in items])
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("pair (sorted by predicted reward difference)"); ax.set_ylabel("reward(chosen) - reward(rejected)")
ax.set_title(f"Reward model vs judge label on {len(items)} pairs (orange = held-out, blue = train)")
plt.tight_layout(); plt.show()
print("agreement with label: held-out", f"{int((hold_diff > 0).sum())}/{len(hold_diff)}", "| shown train pairs",
      f"{int(sum(v > 0 for v, s in items if s == 'train'))}/{sum(s == 'train' for _, s in items)}")
checkpoint("חלק ד׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק ד׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק ה׳ — DPO</p>
<ul>
<li>סעיף 14: DPO על מודל ה-SFT, LoRA מעליו, beta=0.1</li>
<li>סעיף 15: DPO מול SFT על הסט הקבוע ועל סט השופט</li>
<li>מעגליות: אותן 100 השוואות גם בשופט שני, והסכמה בין השופטים</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>דגימת 8 מועמדים לכל פרומפט בשתי הגרלות בלתי תלויות, וניקוד. כל השוואה בחלק ו׳ נשענת על 100.</p>
</div>

In [ ]:
free_judge()
sft = None
free_gpu()
dpo_train = Dataset.from_list([dict(prompt=p["prompt"], chosen=p["chosen"] + "<|im_end|>", rejected=p["rejected"] + "<|im_end|>") for p in pref_train])
dpo_eval = Dataset.from_list([dict(prompt=p["prompt"], chosen=p["chosen"] + "<|im_end|>", rejected=p["rejected"] + "<|im_end|>") for p in pref_hold])
policy = load_sft(SFT_DIR)
dpo_steps = math.ceil(len(dpo_train) / (CFG["dpo_bsz"] * CFG["dpo_grad_acc"])) * CFG["dpo_epochs"]
dpo_args = DPOConfig(
    output_dir=os.path.join(RUN_DIR, "dpo"), beta=CFG["dpo_beta"], num_train_epochs=CFG["dpo_epochs"],
    per_device_train_batch_size=CFG["dpo_bsz"], per_device_eval_batch_size=CFG["dpo_bsz"],
    gradient_accumulation_steps=CFG["dpo_grad_acc"], learning_rate=CFG["dpo_lr"], lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.05 * dpo_steps)), optim="adamw_torch", bf16=BF16, fp16=(DEVICE == "cuda" and not BF16),
    max_length=CFG["max_len"] + CFG["max_new"], logging_steps=max(1, dpo_steps // 10), logging_first_step=True,
    eval_strategy="epoch", save_strategy="no", report_to="none", seed=GLOBAL_SEED, dataloader_pin_memory=False,
    use_cpu=(DEVICE == "cpu"), disable_tqdm=False,
)
dpo_trainer = DPOTrainer(model=policy, ref_model=None, args=dpo_args, train_dataset=dpo_train, eval_dataset=dpo_eval,
                         processing_class=tok, peft_config=LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"],
                                                                      lora_dropout=CFG["lora_dropout"], target_modules=LORA_ALL, task_type="CAUSAL_LM"))
n_tr, _ = count_params(dpo_trainer.model)
free_gpu()
out = dpo_trainer.train()
hist = dpo_trainer.state.log_history
train_log = [{k: round(v, 4) for k, v in h.items() if k in ("step", "loss", "rewards/accuracies", "rewards/margins")} for h in hist if "loss" in h]
eval_log = [{k: round(v, 4) for k, v in h.items() if k in ("epoch", "eval_loss", "eval_rewards/accuracies", "eval_rewards/margins")} for h in hist if "eval_loss" in h]
RESULTS["runs"]["dpo"] = dict(method="dpo_lora_on_sft", base=SFT_DIR, beta=CFG["dpo_beta"], epochs=CFG["dpo_epochs"], steps=dpo_steps,
                              lr=CFG["dpo_lr"], effective_batch=CFG["dpo_bsz"] * CFG["dpo_grad_acc"], n_pairs=len(dpo_train),
                              trainable=n_tr, trainable_pct=round(100 * n_tr / N_TOTAL_BASE, 4),
                              peak_train_gib=(gib(torch.cuda.max_memory_allocated()) if DEVICE == "cuda" else None),
                              train_runtime_s=round(out.metrics["train_runtime"], 1), seed=GLOBAL_SEED,
                              dtype=f"{ENV['dtype']} SFT weights, float32 LoRA adapters", train_log=train_log, eval_log=eval_log)
dpo_model = dpo_trainer.model
print("train log:", train_log[-3:])
print("eval log :", eval_log)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מהלך אימון ה-DPO בגרף.</p>
</div>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
st = [h["step"] for h in train_log]
ax[0].plot(st, [h.get("loss") for h in train_log], marker="o", label="train loss")
ax[0].axhline(math.log(2), color="k", ls="--", lw=1, label="log 2, no preference yet")
ax[0].set_xlabel("step"); ax[0].set_ylabel("DPO loss"); ax[0].set_title("DPO loss"); ax[0].legend(fontsize=8)
ax[1].plot(st, [h.get("rewards/margins") for h in train_log], marker="o", label="train pairs")
if eval_log:
    ep = np.array([h["epoch"] for h in eval_log], dtype=float)
    ax[1].plot(ep * max(st) / max(ep.max(), 1e-9), [h.get("eval_rewards/margins") for h in eval_log], marker="s", label="held-out pairs")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xlabel("step"); ax[1].set_ylabel("reward margin"); ax[1].set_title("implicit reward margin, chosen minus rejected")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תשובות מודל ה-DPO.</p>
</div>

In [ ]:
gen_eval(dpo_model, "dpo")
stats_e = pd.DataFrame([behavior_stats(resp["full_ft"]), behavior_stats(resp["dpo"])], index=["sft (greedy)", "dpo (greedy)"])
display(stats_e)
display(check_table(["full_ft", "dpo"]))
for f, a, b in list(zip(FIXED, resp["full_ft"], resp["dpo"]))[:3]:
    print(f"--- {f['id']}\nSFT: {a['text'][:220]}\nDPO: {b['text'][:220]}\n")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>השופט הראשי והשופט השני, DPO מול SFT.</p>
</div>

In [ ]:
d_e1, s_e1 = compare_judgeset("E_dpo_vs_sft", "dpo", "full_ft")
compare_fixed("E_dpo_vs_sft_fixed", "dpo", "full_ft")
d_e2, s_e2 = compare_judgeset("E_dpo_vs_sft_judge2", "dpo", "full_ft", name=JUDGE2_MODEL)
free_judge()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>cohen_kappa: הסכמה בין שופטים מעבר למקרה.</p>
</div>

In [ ]:
def cohen_kappa(a, b):
    labels = sorted(set(a) | set(b))
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(l) / n) * (b.count(l) / n) for l in labels)
    return round((po - pe) / (1 - pe), 3) if pe < 1 else float("nan"), round(po, 3)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הסכמת השופטים.</p>
</div>

In [ ]:
kappa, agree = cohen_kappa(list(d_e1.result), list(d_e2.result))
RESULTS["part_e"] = dict(judge_agreement=dict(kappa=kappa, raw_agreement=agree, n=len(d_e1)), behavior=stats_e.to_dict("index"))
print(f"judge agreement on {len(d_e1)} pairs: raw {agree} | Cohen's kappa {kappa}")
display(summary_table(["E_dpo_vs_sft", "E_dpo_vs_sft_fixed", "E_dpo_vs_sft_judge2"]))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>תוצאות השופטים ומטריצת ההסכמה.</p>
</div>

In [ ]:
plot_outcomes(["E_dpo_vs_sft", "E_dpo_vs_sft_fixed", "E_dpo_vs_sft_judge2"], "Part E: DPO vs SFT")
labs = ["x", "tie", "y"]
cm = pd.crosstab(pd.Categorical(d_e1.result, labs), pd.Categorical(d_e2.result, labs), dropna=False)
fig, ax = plt.subplots(figsize=(4.8, 3.9))
ax.imshow(cm.values, cmap="Blues")
names = ["DPO wins", "tie", "SFT wins"]
ax.set_xticks(range(3)); ax.set_xticklabels(names); ax.set_yticks(range(3)); ax.set_yticklabels(names)
ax.set_xlabel("judge 2"); ax.set_ylabel("judge 1")
for i in range(3):
    for j in range(3):
        ax.text(j, i, int(cm.values[i, j]), ha="center", va="center")
ax.set_title(f"judge agreement, Cohen's kappa {kappa}")
plt.tight_layout(); plt.show()
checkpoint("חלק ה׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק ה׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חלק ו׳ — Best-of-N</p>
<ul>
<li>סעיף 16: 8 מועמדים מה-SFT לכל פרומפט, ה-reward model בוחר, N=4 ו-N=8</li>
<li>סעיף 17: שופט מול דגימה בודדת ומול DPO, ועלות לשאילתה</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>דגימת 8 מועמדים לכל פרומפט בשתי הגרלות בלתי תלויות, וניקוד. כל השוואה בחלק ו׳ נשענת על 100.</p>
</div>

In [ ]:
free_judge()
free_gpu()
sft = load_sft(SFT_DIR)
N = CFG["bon_N"]
BON_SEEDS = [GLOBAL_SEED + 99 + k for k in range(len(CFG["eval_seeds"]))]
cands, bon_time = [], 0.0
for sd in BON_SEEDS:
    bon_out, t = generate(sft, JUDGE_PROMPTS, temperature=CFG["eval_temp"], top_p=CFG["pref_top_p"],
                          seed=sd, num_return=N, batch_size=max(1, CFG["judge_bsz"] // 2))
    bon_time += t
    cands += [bon_out[i * N:(i + 1) * N] for i in range(len(JUDGE_PROMPTS))]
BON_PROMPTS = JUDGE_PROMPTS * len(BON_SEEDS)
BON_INSTR = JUDGE_INSTR * len(BON_SEEDS)
t0 = time.perf_counter()
scores = [rm_score(embed_responses(sft, [p] * len(c), [x["text"] for x in c])) for p, c in zip(BON_PROMPTS, cands)]
score_time = (time.perf_counter() - t0) / len(BON_PROMPTS)
print(f"{len(BON_SEEDS)} draws x {len(JUDGE_PROMPTS)} prompts x {N} candidates in {bon_time:.1f}s | scoring {score_time:.2f}s per prompt")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מדידת עלות לשאילתה בודדת, N=1, 4, 8.</p>
</div>

In [ ]:
cost = {1: [], 4: [], CFG["bon_N"]: []}
for p in FIXED_PROMPTS:
    for n in cost:
        o, t = generate(sft, [p], temperature=CFG["eval_temp"], top_p=CFG["pref_top_p"], seed=GLOBAL_SEED + 99, num_return=n, batch_size=1)
        cost[n].append(dict(time_s=t, tokens=sum(x["n_tokens"] for x in o)))
cost_tab = pd.DataFrame({f"N={n}": dict(gen_time_s=round(float(np.mean([c["time_s"] for c in v])), 2),
                                        tokens=round(float(np.mean([c["tokens"] for c in v])), 1),
                                        score_time_s=(0.0 if n == 1 else round(score_time * n / CFG["bon_N"], 2)))
                         for n, v in cost.items()}).T
cost_tab["total_time_s"] = (cost_tab.gen_time_s + cost_tab.score_time_s).round(2)
display(cost_tab)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בחירת המועמד הטוב וגרף הציון מול N.</p>
</div>

In [ ]:
Ns = [n for n in [1, 2, 4, 8] if n <= CFG["bon_N"]]
curve = {n: float(np.mean([max(s[:n]) for s in scores])) for n in Ns}
BON = {"sft_single": [c[0] for c in cands]}
for n in dict.fromkeys([4, N]):
    BON[f"bon{n}"] = [c[int(np.argmax(s[:n]))] for c, s in zip(cands, scores)]
BON_TOP = f"bon{N}"
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(list(curve), list(curve.values()), marker="o"); ax.set_xscale("log", base=2)
ax.set_xlabel("N candidates"); ax.set_ylabel("mean reward of selected candidate"); ax.set_title("Best-of-N on the SFT model")
plt.tight_layout(); plt.show()
print({n: round(v, 3) for n, v in curve.items()})
RESULTS["part_f"] = dict(reward_vs_N=curve, cost=cost_tab.to_dict("index"), n_prompts=len(cands))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>השופט, 100 השוואות: Best-of-4 ו-Best-of-8 מול דגימה בודדת, ו-Best-of-8 מול DPO.</p>
</div>

In [ ]:
BT = lambda k: [o["text"] for o in BON[k]]
compare("F_bon4_vs_sft", BON_INSTR, BT("bon4"), BT("sft_single"), "bon4", "sft_single")
compare("F_bon8_vs_sft", BON_INSTR, BT(BON_TOP), BT("sft_single"), BON_TOP, "sft_single")
compare("F_bon8_vs_dpo", BON_INSTR, BT(BON_TOP), texts(JSAMP, "dpo"), BON_TOP, "dpo_single")
display(summary_table(["F_bon4_vs_sft", "F_bon8_vs_sft", "F_bon8_vs_dpo"]))
stats_f = pd.DataFrame([dict(n=len(v), mean_words=round(float(np.mean([len(words(o["text"])) for o in v])), 1),
                             ended_rate=round(float(np.mean([o["ended"] for o in v])), 3),
                             rep4=round(float(np.mean([rep4(o["text"]) for o in v])), 3)) for v in BON.values()],
                       index=list(BON))
display(stats_f)
RESULTS["part_f"]["behavior"] = stats_f.to_dict("index")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרפים לחלק ו׳: תוצאות השופט, וציון מול עלות.</p>
</div>

In [ ]:
plot_outcomes(["F_bon4_vs_sft", "F_bon8_vs_sft", "F_bon8_vs_dpo"], "Part F: Best-of-N")
fig, ax = plt.subplots(figsize=(6.5, 3.4))
pts = [(cost_tab.loc[k, "total_time_s"], curve.get(int(k.split("=")[1]), np.nan), k) for k in cost_tab.index]
ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o")
for x, yv, k in pts:
    ax.annotate(k, (x, yv), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("seconds per query, single query"); ax.set_ylabel("mean reward of the selected answer")
ax.set_title("reward bought by extra inference time")
plt.tight_layout(); plt.show()
checkpoint("חלק ו׳")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — חלק ו׳:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בונוס — מיקום שכבות LoRA</p>
<ul>
<li>סעיף 23: LoRA על q, k, v, o בלבד, מול כל השכבות הליניאריות, באותם תנאים</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>אימון LoRA על attention בלבד.</p>
</div>

In [ ]:
m = run_sft("lora_attn", "lora", train_ds, eval_ds, targets=LORA_ATTN)
gen_eval(m, "lora_attn")
m = None
free_gpu()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>טבלאות, מדדים ושופט.</p>
</div>

In [ ]:
tab_bonus = run_table(["lora", "lora_attn"])
display(tab_bonus[["trainable", "trainable_pct", "peak_gib", "time_s", "train_loss", "eval_loss"]])
stats_bonus = pd.DataFrame([behavior_stats(resp[k]) for k in ["lora", "lora_attn"]], index=["lora (all-linear)", "lora_attn"])
display(stats_bonus)
compare_fixed("BONUS_attn_vs_base", "lora_attn", "base")
compare_judgeset("BONUS_attn_vs_all", "lora_attn", "lora")
display(summary_table(["BONUS_attn_vs_base", "BONUS_attn_vs_all", "B_lora_vs_base"]))
RESULTS["bonus"] = dict(runs=tab_bonus.to_dict("index"), behavior=stats_bonus.to_dict("index"))
free_judge()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרפים לבונוס.</p>
</div>

In [ ]:
plot_losses(["lora", "lora_attn"], "Bonus: LoRA on all linear layers vs attention only")
plot_outcomes(["BONUS_attn_vs_base", "BONUS_attn_vs_all"], "Bonus: judge outcomes")
checkpoint("חלק הבונוס")

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ממצאים — בונוס:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>סינתזה — פרופיל אימון ויישור</p>
<ul>
<li>סעיף 18: טבלה של כל השיטות: איכות לפי שופט, זמן אימון, זיכרון, עלות הסקה</li>
<li>win-rate הוא ניצחונות מכל ההשוואות, כנדרש. pref_decisive הוא ניצחונות מתוך ההכרעות, והוא מראה כיוון</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בניית טבלת הסינתזה והגרפים.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>wr: win-rate, שיעור תיקו ומשווה של השוואה אחת.</p>
</div>

In [ ]:
def wr(label):
    s = RESULTS["comparisons"].get(label, {}).get("summary")
    return (s["win_rate"], s["tie_rate"], s["y"]) if s else ((None, None, None), None, None)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>run_get: ערך אחד מרשומת אימון.</p>
</div>

In [ ]:
def run_get(name, key):
    return RESULTS["runs"].get(name, {}).get(key)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הטבלה והגרפים.</p>
</div>

In [ ]:
cost_f = RESULTS["part_f"]["cost"]
rows = []
for method, label, run, infer in [("Full FT", "B_full_vs_base_js", "full_ft", "N=1"), ("LoRA", "B_lora_vs_base_js", "lora", "N=1"),
                                  ("QLoRA", "B_qlora_vs_base_js", "qlora", "N=1"), ("DPO (LoRA on SFT)", "E_dpo_vs_sft", "dpo", "N=1"),
                                  ("Best-of-4", "F_bon4_vs_sft", None, "N=4"), (f"Best-of-{CFG['bon_N']}", "F_bon8_vs_sft", None, f"N={CFG['bon_N']}")]:
    if any(r["method"] == method for r in rows):
        continue
    (p, lo, hi), tie, comp = wr(label)
    s = RESULTS["comparisons"].get(label, {}).get("summary", {})
    dp = s.get("pref_decisive") or (None, None, None)
    rows.append(dict(method=method, compared_to=comp, n=s.get("n"), win_rate=p, ci_low=lo, ci_high=hi, tie_rate=tie,
                     pref_decisive=dp[0], pref_low=dp[1], pref_high=dp[2],
                     trainable_pct=(run_get(run, "trainable_pct") if run else 0.0),
                     train_time_s=(run_get(run, "train_runtime_s") if run else 0.0),
                     peak_train_gib=(run_get(run, "peak_train_gib") if run else None),
                     infer_time_s=cost_f[infer]["total_time_s"], infer_tokens=cost_f[infer]["tokens"]))
synth = pd.DataFrame(rows).set_index("method")
display(synth)
RESULTS["synthesis"] = synth.to_dict("index")
plot_outcomes(["B_full_vs_base_js", "B_lora_vs_base_js", "B_qlora_vs_base_js", "E_dpo_vs_sft", "F_bon4_vs_sft", "F_bon8_vs_sft"],
              "every method against its comparison, same judge")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].barh(synth.index, synth.train_time_s.fillna(0)); ax[0].set_title("training time"); ax[0].invert_yaxis()
ax[0].set_xlabel("seconds"); ax[0].set_ylabel("method")
ax[1].barh(synth.index, synth.infer_time_s); ax[1].set_title("inference time per query"); ax[1].invert_yaxis()
ax[1].set_xlabel("seconds"); ax[1].set_ylabel("method")
plt.tight_layout(); plt.show()

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>פרופיל אימון ויישור — עמוד הסיכום, סעיפים 19 ו-20:</p>
<p>להשלים אחרי ההרצה.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מגבלות, כנדרש בנספח ב׳:</p>
<ul>
<li>מעגליות: אותו שופט תייג את חלק ד׳ ומדד את ה׳ ו-ו׳. השופט השני מקהה, לא מבטל</li>
<li>מאות בודדות של זוגות העדפה. מעט ממה שנהוג</li>
<li>seed אימון אחד. בהרצות קודמות של אותו קוד, השוואה אחת נתנה מסקנה שונה בין הרצות</li>
<li>זמנים ב-Colab השתנו עד פי 2 בין הרצות קודמות. זיכרון יציב</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שמירת כל התוצאות לקובץ.</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>jsonable: המרת ערכי numpy ל-JSON.</p>
</div>

In [ ]:
def jsonable(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>כתיבת הקובץ.</p>
</div>

In [ ]:
RESULTS["responses"] = {k: [dict(id=f["id"], **o) for f, o in zip(FIXED, v)] for k, v in resp.items()}
RESULTS["responses_sampled"] = {f"{k[0]}_seed{k[1]}": [o["text"] for o in v] for k, v in SAMP.items()}
RESULTS["responses_judgeset"] = {f"{k[0]}_seed{k[1]}": [o["text"] for o in v] for k, v in JSAMP.items()}
RESULTS["best_of_n"] = {k: [o["text"] for o in v] for k, v in BON.items()}
RESULTS["preference_pairs"] = dict(train=pref_train, holdout=pref_hold)
with open(os.path.join(RUN_DIR, "hw2_results.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=1, ensure_ascii=False, default=jsonable)
print("saved", os.path.join(RUN_DIR, "hw2_results.json"), f"{os.path.getsize(os.path.join(RUN_DIR, 'hw2_results.json')) / 1e6:.2f} MB")
print("runs:", list(RESULTS["runs"]))
print("comparisons:", list(RESULTS["comparisons"]))

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>README:</p>
<ul>
<li>הרצה: Colab, GPU יחיד (L4 מומלץ), Runtime ואז Run all</li>
<li>אין צורך במפתחות. כל המודלים והדאטה פתוחים</li>
<li>מודלים: Qwen2.5-0.5B, שופט Phi-4 ב-NF4, שופט שני Granite-3.3-8B</li>
<li>זמן ריצה על L4: כ-5 עד 6 שעות</li>
<li>פלטים: hw2_runs/hw2_results.json ומודל ה-SFT ב-hw2_runs/sft_full</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>גרסאות הספריות, גם לקובץ requirements.</p>
</div>

In [ ]:
pkgs = ["torch", "transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes", "scikit-learn", "numpy", "pandas"]
req = []
for p in pkgs:
    try:
        req.append(f"{p}=={im.version(p)}")
    except Exception:
        req.append(f"# {p}: not installed")
with open(os.path.join(RUN_DIR, "requirements.txt"), "w") as f:
    f.write("\n".join(req) + "\n")
print("\n".join(req))
print("python", sys.version.split()[0], "|", ENV["gpu"], "|", ENV["dtype"])

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>נספח: כל התשובות המלאות, דגימות Best-of-N, וזוגות העדפה עם נימוקי השופט.</p>
</div>

In [ ]:
for k in ["base", "full_ft", "lora", "qlora", "lora_small", "lora_noisy", "lora_attn", "dpo"]:
    if k not in resp:
        continue
    print("=" * 100)
    print("MODEL:", k, "(greedy, fixed prompt set)")
    for f, o in zip(FIXED, resp[k]):
        print(f"\n--- [{f['id']}] {f['prompt'][:100]}")
        print(o["text"])
print("=" * 100)
print("BEST-OF-N: the single sample, the Best-of-4 pick and the Best-of-N pick for the first 5 judge-set prompts")
for k in range(min(5, len(JUDGE_INSTR))):
    print(f"\n##### {JUDGE_INSTR[k][:150]}")
    for name in BON:
        print(f"[{name}] {BON[name][k]['text'][:400]}")
print("=" * 100)
print("PREFERENCE PAIRS (first 5 decisive) with judge rationales")
log = {e["i"]: e for e in RESULTS["judge_log"].get("D_pref", []) if e["order"] == "xy"}
log.update({e["i"] + 1000: e for e in RESULTS["judge_log"].get("D_pref_extra", []) if e["order"] == "xy"})
for p in pref_train[:5]:
    print(f"\n### prompt: {p['instruction'][:150]}")
    print("CHOSEN  :", p["chosen"][:400])
    print("REJECTED:", p["rejected"][:400])
    entry = log.get(p["i"])
    print("JUDGE   :", entry["raw"][:400] if entry else "(the judge log for this pair is not available)")